# Exploration Service-Public FPE

Notebook d'exploration pour verifier s'il est possible de sortir d'une selection manuelle des fiches `service-public.fr` en s'appuyant sur le flux XML officiel DILA et sur les metadonnees deja exploitees par le repo.

## Objectifs
- charger l'ensemble du flux XML officiel,
- extraire les metadonnees utiles a la qualification FPE,
- tester une heuristique de detection automatique,
- comparer avec la liste manuelle actuelle,
- produire un compte rendu et un tableau final de recommandation.


In [ ]:
from __future__ import annotations

import json
import traceback
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "scripts" else cwd

import sys
PYTHONPATH_ENTRIES = [
    REPO_ROOT,
    REPO_ROOT / "packages/data-engineering/src",
    REPO_ROOT / "packages/shared-config/src",
]
for entry in reversed(PYTHONPATH_ENTRIES):
    entry_str = str(entry)
    if entry_str not in sys.path:
        sys.path.insert(0, entry_str)

from assistant_rh_data_engineering.service_public.bronze import ServicePublicXmlFetcher
from assistant_rh_data_engineering.service_public.config import BronzeConfig
from assistant_rh_data_engineering.service_public.xml_parser import parse_fiche_xml_from_bytes

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)

MANUAL_CONFIG_PATH = REPO_ROOT / "config" / "service_public_fiches.json"
TARGET_THEME = "travail - formation"
FP_KEYWORDS = [
    "fonction publique",
    "fpe",
    "agent public",
    "agents publics",
    "contractuel",
    "contractuels",
    "fonctionnaire",
    "fonctionnaires",
]

print("Repo root:", REPO_ROOT)
print("Config manuelle:", MANUAL_CONFIG_PATH)


## 1. Chargement du flux XML officiel

Le pipeline du repo sait deja lire le ZIP `vosdroits-latest.zip`. Ici, on le charge sans filtrer sur une liste manuelle de `fiche_ids`.


In [ ]:
fetcher = ServicePublicXmlFetcher(BronzeConfig())
zip_url, zip_bytes = fetcher.download_zip()
assets = list(fetcher.iter_xml_members(zip_bytes))

overview_df = pd.DataFrame(
    [
        {"metric": "ZIP utilise", "value": zip_url},
        {"metric": "Nombre total de fiches XML", "value": len(assets)},
        {"metric": "Exemples d'IDs", "value": ", ".join(fiche_id for fiche_id, _ in assets[:10])},
    ]
)
overview_df


## 2. Parsing et extraction des metadonnees

On parse toutes les fiches et on extrait les champs qui peuvent servir a qualifier une fiche FPE:
- `theme`,
- `situations`,
- `title`,
- `subtitles`,
- `context`,
- quelques indicateurs de structure.


In [ ]:
rows = []
parse_errors = []

for idx, (fiche_id, xml_bytes) in enumerate(assets, start=1):
    try:
        parsed = parse_fiche_xml_from_bytes(xml_bytes, fiche_id)
        if not parsed:
            parse_errors.append(
                {
                    "fiche_id": fiche_id,
                    "error_type": "parse_none",
                    "message": "Le parseur a renvoye None.",
                }
            )
            continue

        meta = parsed.get("metadata", {}) or {}
        situations = meta.get("situations", []) or []

        rows.append(
            {
                "fiche_id": fiche_id,
                "title": parsed.get("title", "") or "",
                "source_url": parsed.get("source_url", "") or "",
                "theme": meta.get("theme", "") or "",
                "category": meta.get("category", "") or "",
                "situations": situations,
                "subtitles": meta.get("subtitles", "") or "",
                "context": meta.get("context", "") or "",
                "date_verification": meta.get("date_verification", "") or "",
                "last_updated_date": parsed.get("last_updated_date", "") or "",
                "has_references_juridiques": bool(meta.get("references_juridiques")),
                "references_juridiques_count": len(meta.get("references_juridiques", []) or []),
                "char_count": parsed.get("char_count", 0) or 0,
                "token_count": parsed.get("token_count", 0) or 0,
            }
        )
    except Exception as exc:
        parse_errors.append(
            {
                "fiche_id": fiche_id,
                "error_type": type(exc).__name__,
                "message": str(exc),
                "traceback": traceback.format_exc(limit=1),
            }
        )

df = pd.DataFrame(rows)
errors_df = pd.DataFrame(parse_errors)

quality_df = pd.DataFrame(
    [
        {"metric": "Fiches parsees avec succes", "value": len(df)},
        {"metric": "Erreurs de parsing", "value": len(errors_df)},
        {"metric": "Fiches avec situations", "value": int(df["situations"].apply(bool).sum())},
        {"metric": "Fiches avec references juridiques", "value": int(df["has_references_juridiques"].sum())},
    ]
)
quality_df


## 3. Exploration des signaux disponibles

On regarde d'abord les themes et situations les plus frequents pour verifier si `Travail - Formation` et `FPE` sont de bons signaux a exploiter.


In [ ]:
def normalize_situations(values):
    return [str(v).strip().upper() for v in (values or []) if str(v).strip()]

df["theme_clean"] = df["theme"].fillna("").str.strip()
df["situations_norm"] = df["situations"].apply(normalize_situations)

theme_counts = (
    df.groupby("theme_clean", dropna=False)
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)

all_situations = []
for values in df["situations_norm"]:
    all_situations.extend(values)

situation_counts = pd.DataFrame(
    Counter(all_situations).most_common(),
    columns=["situation", "count"],
)

display(Markdown("### Themes les plus frequents"))
display(theme_counts.head(20))
display(Markdown("### Situations les plus frequentes"))
display(situation_counts)


## 4. Heuristique de detection FPE

Heuristique testee dans ce notebook:

- `keep_strong` : la fiche expose explicitement `FPE` dans `situations`.
- `exclude_other_fp` : la fiche expose seulement `FPT` / `FPH` mais pas `FPE`.
- `keep_probable` : le theme est `Travail - Formation` et le titre / contexte contient des mots-cles fonction publique.
- `review` : des mots-cles fonction publique sont presents, mais le signal est moins robuste.
- `exclude` : aucun signal suffisant.


In [ ]:
def normalize_text(*parts):
    return " ".join(str(p or "") for p in parts).lower().strip()

def extract_text_blob(row):
    return normalize_text(
        row.get("title", ""),
        row.get("subtitles", ""),
        row.get("context", ""),
    )

def detect_fp_keywords(row):
    text = extract_text_blob(row)
    return [kw for kw in FP_KEYWORDS if kw in text]

def classify_row(row):
    situations = set(row.get("situations_norm", []))
    theme = str(row.get("theme", "") or "").strip().lower()
    matched_keywords = detect_fp_keywords(row)
    has_fp_keywords = bool(matched_keywords)

    if "FPE" in situations:
        return "keep_strong"

    if situations and "FPE" not in situations and situations.issubset({"FPT", "FPH"}):
        return "exclude_other_fp"

    if theme == TARGET_THEME and has_fp_keywords:
        return "keep_probable"

    if has_fp_keywords:
        return "review"

    return "exclude"

df["matched_keywords"] = df.apply(detect_fp_keywords, axis=1)
df["has_fp_keywords"] = df["matched_keywords"].apply(bool)
df["decision"] = df.apply(classify_row, axis=1)

decision_counts = (
    df.groupby("decision")
      .size()
      .reset_index(name="count")
      .sort_values(["count", "decision"], ascending=[False, True])
)
decision_counts


## 5. Comparaison avec la liste manuelle actuelle

On compare la detection automatique a `config/service_public_fiches.json` pour identifier:
- les fiches deja couvertes,
- les nouvelles candidates,
- les fiches de la liste manuelle que l'heuristique ne retrouve pas.


In [ ]:
manual_config = json.loads(MANUAL_CONFIG_PATH.read_text(encoding="utf-8"))
manual_ids = set(manual_config.get("fiche_ids", []))

auto_strong_ids = set(df[df["decision"].isin(["keep_strong", "keep_probable"])]["fiche_id"])
auto_with_review_ids = set(df[df["decision"].isin(["keep_strong", "keep_probable", "review"])]["fiche_id"])

shared_ids = sorted(manual_ids & auto_with_review_ids)
new_candidates_vs_manual = sorted(auto_strong_ids - manual_ids)
manual_only_ids = sorted(manual_ids - auto_with_review_ids)

comparison_df = pd.DataFrame(
    [
        {"metric": "Liste manuelle actuelle", "value": len(manual_ids)},
        {"metric": "Auto keep_strong + keep_probable", "value": len(auto_strong_ids)},
        {"metric": "Auto incluant review", "value": len(auto_with_review_ids)},
        {"metric": "Intersection avec la liste manuelle", "value": len(shared_ids)},
        {"metric": "Nouvelles fiches candidates vs liste manuelle", "value": len(new_candidates_vs_manual)},
        {"metric": "Fiches manuelles non retrouvees", "value": len(manual_only_ids)},
    ]
)
comparison_df


In [ ]:
candidate_cols = [
    "fiche_id",
    "title",
    "theme",
    "situations_norm",
    "matched_keywords",
    "decision",
    "subtitles",
    "context",
]

strong_candidates_df = (
    df[df["decision"].isin(["keep_strong", "keep_probable"])][candidate_cols]
      .sort_values(["decision", "theme", "title"])
      .reset_index(drop=True)
)

new_candidates_df = (
    df[df["fiche_id"].isin(new_candidates_vs_manual)][candidate_cols]
      .sort_values(["decision", "theme", "title"])
      .reset_index(drop=True)
)

manual_only_df = (
    df[df["fiche_id"].isin(manual_only_ids)][candidate_cols]
      .sort_values(["theme", "title"])
      .reset_index(drop=True)
)

review_df = (
    df[df["decision"] == "review"][candidate_cols]
      .sort_values(["theme", "title"])
      .reset_index(drop=True)
)

display(Markdown("### Extrait des fiches retenues automatiquement"))
display(strong_candidates_df.head(30))
display(Markdown("### Extrait des nouvelles candidates absentes de la liste manuelle"))
display(new_candidates_df.head(30))
display(Markdown("### Extrait des fiches manuelles non retrouvees"))
display(manual_only_df.head(30))


## 6. Tableau final de decision

Ce tableau synthese positionne les differents signaux et la recommandation associee.


In [ ]:
final_decision_table = pd.DataFrame(
    [
        {
            "signal": "situations contient FPE",
            "robustesse": "elevee",
            "couverture": "bonne quand le XML expose des onglets FPE/FPT/FPH",
            "effort_humain": "faible",
            "usage_recommande": "garde automatique",
        },
        {
            "signal": "theme = Travail - Formation + mots-cles FP",
            "robustesse": "moyenne",
            "couverture": "utile pour completer les fiches sans situations explicites",
            "effort_humain": "modere",
            "usage_recommande": "garde probable",
        },
        {
            "signal": "mots-cles FP seulement",
            "robustesse": "faible a moyenne",
            "couverture": "detecte des candidates mais avec plus de bruit",
            "effort_humain": "cible",
            "usage_recommande": "revue humaine",
        },
        {
            "signal": "liste manuelle uniquement",
            "robustesse": "dependante de la connaissance metier",
            "couverture": "faible a moyen terme",
            "effort_humain": "eleve",
            "usage_recommande": "a remplacer",
        },
    ]
)
final_decision_table


## 7. Compte rendu final

Le texte ci-dessous est genere a partir des resultats du notebook et peut servir de base a un message de synthese.


In [ ]:
stats = {
    "total_fiches_parsees": len(df),
    "parse_errors": len(errors_df),
    "fiches_avec_situations": int(df["situations_norm"].apply(bool).sum()),
    "fiches_avec_FPE": int(df["situations_norm"].apply(lambda xs: "FPE" in xs).sum()),
    "fiches_travail_formation": int(df["theme"].str.lower().eq(TARGET_THEME).sum()),
    "fiches_avec_mots_cles_fp": int(df["has_fp_keywords"].sum()),
    "keep_strong": int((df["decision"] == "keep_strong").sum()),
    "keep_probable": int((df["decision"] == "keep_probable").sum()),
    "review": int((df["decision"] == "review").sum()),
    "exclude_other_fp": int((df["decision"] == "exclude_other_fp").sum()),
    "exclude": int((df["decision"] == "exclude").sum()),
}

report_lines = []
report_lines.append("# Compte rendu d'exploration Service-Public FPE")
report_lines.append("")
report_lines.append("## Resultats cles")
report_lines.append(f"- ZIP officiel utilise : `{zip_url}`")
report_lines.append(f"- Nombre total de fiches parsees : **{stats['total_fiches_parsees']}**")
report_lines.append(f"- Erreurs de parsing : **{stats['parse_errors']}**")
report_lines.append(f"- Fiches avec `situations` : **{stats['fiches_avec_situations']}**")
report_lines.append(f"- Fiches contenant explicitement `FPE` : **{stats['fiches_avec_FPE']}**")
report_lines.append(f"- Fiches du theme `Travail - Formation` : **{stats['fiches_travail_formation']}**")
report_lines.append(f"- Fiches avec mots-cles fonction publique : **{stats['fiches_avec_mots_cles_fp']}**")
report_lines.append("")
report_lines.append("## Lecture de l'heuristique")
report_lines.append(f"- `keep_strong` : **{stats['keep_strong']}**")
report_lines.append(f"- `keep_probable` : **{stats['keep_probable']}**")
report_lines.append(f"- `review` : **{stats['review']}**")
report_lines.append(f"- `exclude_other_fp` : **{stats['exclude_other_fp']}**")
report_lines.append(f"- `exclude` : **{stats['exclude']}**")
report_lines.append("")
report_lines.append("## Comparaison avec la liste manuelle actuelle")
report_lines.append(f"- Taille de la liste manuelle actuelle : **{len(manual_ids)}**")
report_lines.append(f"- Fiches detectees automatiquement (fort + probable) : **{len(auto_strong_ids)}**")
report_lines.append(f"- Fiches detectees automatiquement (incluant review) : **{len(auto_with_review_ids)}**")
report_lines.append(f"- Intersection avec la liste manuelle : **{len(shared_ids)}**")
report_lines.append(f"- Nouvelles fiches candidates absentes de la liste manuelle : **{len(new_candidates_vs_manual)}**")
report_lines.append(f"- Fiches manuelles non retrouvees par l'heuristique : **{len(manual_only_ids)}**")
report_lines.append("")
report_lines.append("## Conclusion")
report_lines.append("- La source officielle est deja exhaustive ; le point limitant actuel est la selection manuelle des `fiche_ids`.")
report_lines.append("- Le meilleur signal operationnel est la presence explicite de `FPE` dans `situations`.")
report_lines.append("- `theme = Travail - Formation` et les mots-cles fonction publique sont utiles comme complements, mais pas comme critere unique d'exhaustivite.")
report_lines.append("- La bonne strategie semble etre : garde automatique sur `FPE`, completement par heuristique `theme + keywords`, puis revue humaine ciblee sur les cas `review`.")

report_text = "\n".join(report_lines)
display(Markdown(report_text))
report_text
